# eICU Multi-Center Validation

In [ ]:
import os, numpy as np, pandas as pd
from itertools import permutations, combinations
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from scipy import stats
from xgboost import XGBClassifier

RANDOM_STATE=42; N_FOLDS=5
QUARTILES=['Q1','Q2','Q3','Q4']
MORTALITY='hospital_expire_flag'
FREQ_COL='average_item_interval'

# PRIMARY feature set: eICU-native APACHE IVa physiology (apacheapsvar, aps_ prefix).
# Treatment cols excluded
APS_PRIMARY=['aps_eyes','aps_motor','aps_verbal','aps_urine','aps_wbc','aps_temperature',
 'aps_respiratoryrate','aps_sodium','aps_heartrate','aps_meanbp','aps_ph','aps_hematocrit',
 'aps_creatinine','aps_albumin','aps_pao2','aps_pco2','aps_bun','aps_glucose','aps_bilirubin','aps_fio2']

# XGBoost hyperparameters 
XGB_PARAMS=dict(
    max_depth=3,            
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,    # >=10 patients per leaf
    gamma=1.0,              # min loss reduction to split
    reg_alpha=0.5,          # L1
    reg_lambda=2.0,         # L2
    n_estimators=500,       # upper bound; early stopping finds optimum
    eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1)
EARLY_STOP_ROUNDS=20; VAL_FRACTION=0.15
MIN_RACE_N, MIN_RACE_DEATHS = 40, 10

MIT=r""
PHENOTYPES={'Glucose':'feats_glucose.csv','Pain':'feats_pain.csv'}
# Per-phenotype APACHE window (obtained from deccomposition)
WINDOWS={'Glucose':(50,125), 'Pain':(50,95)}
print("setup ready; xgboost", __import__('xgboost').__version__)

## Training (two-phase early stopping + Platt) & stats

In [ ]:
def train_xgb(X, y):
    spw=float((y==0).sum())/max(float((y==1).sum()),1.0)
    Xtr,Xval,ytr,yval=train_test_split(X,y,test_size=VAL_FRACTION,stratify=y,random_state=RANDOM_STATE)
    probe=XGBClassifier(**{**XGB_PARAMS,'scale_pos_weight':spw,'early_stopping_rounds':EARLY_STOP_ROUNDS})
    probe.fit(Xtr,ytr,eval_set=[(Xval,yval)],verbose=False)
    bn=getattr(probe,'best_iteration',None); bn=(bn+1) if bn is not None else XGB_PARAMS['n_estimators']
    final=XGBClassifier(**{**XGB_PARAMS,'scale_pos_weight':spw,'n_estimators':bn})
    final.fit(X,y,verbose=False); return final

def fit_platt_holdout(model,X,y):
    raw=model.predict_proba(X)[:,1]; lr=LogisticRegression(); lr.fit(raw.reshape(-1,1),y); return lr
def apply_platt(lr,raw): return lr.predict_proba(raw.reshape(-1,1))[:,1]

def boot_ci(y,p,n_boot=2000,seed=RANDOM_STATE):
    y,p=np.asarray(y),np.asarray(p)
    if len(y)<5 or len(np.unique(y))<2: return (np.nan,np.nan,np.nan)
    rng=np.random.default_rng(seed); a=[]
    for _ in range(n_boot):
        i=rng.integers(0,len(y),len(y))
        if len(np.unique(y[i]))<2: continue
        a.append(roc_auc_score(y[i],p[i]))
    if not a: return (np.nan,np.nan,np.nan)
    a=np.array(a); return (roc_auc_score(y,p),float(np.percentile(a,2.5)),float(np.percentile(a,97.5)))

def _midrank(x):
    J=np.argsort(x); Z=x[J]; N=len(x); T=np.zeros(N); i=0
    while i<N:
        j=i
        while j<N and Z[j]==Z[i]: j+=1
        T[i:j]=0.5*(i+j-1)+1; i=j
    out=np.empty(N); out[J]=T; return out
def _auc_cov(y,p):
    y=np.asarray(y); p=np.asarray(p); pos=p[y==1]; neg=p[y==0]; m,n=len(pos),len(neg)
    if m==0 or n==0: return np.nan,np.nan
    tx=_midrank(pos); ty=_midrank(neg); tz=_midrank(np.concatenate([pos,neg]))
    auc=(tz[:m].sum()-m*(m+1)/2)/(m*n); v01=(tz[:m]-tx)/n; v10=1-(tz[m:]-ty)/m
    return auc, v01.var(ddof=1)/m+v10.var(ddof=1)/n
def delong_p(yA,pA,yB,pB):
    aA,sA=_auc_cov(yA,pA); aB,sB=_auc_cov(yB,pB)
    if np.isnan(sA) or np.isnan(sB) or (sA+sB)==0: return np.nan
    z=(aA-aB)/np.sqrt(sA+sB); return 2*(1-stats.norm.cdf(abs(z)))

def _jt_stat(groups,order):
    go=[groups[i] for i in order]; s=0.0
    for a in range(len(go)):
        for b in range(a+1,len(go)):
            for u in go[a]:
                for v in go[b]: s+=(u>v)+0.5*(u==v)
    return s
def perm_max_jt_general(per_fold_list,n_perm=1000,seed=RANDOM_STATE):
    groups=[np.asarray(g) for g in per_fold_list if len(g)>0]; k=len(groups)
    if k<2: return np.nan
    orders=list(permutations(range(k))); obs=max(_jt_stat(groups,o) for o in orders)
    allv=np.concatenate(groups); sizes=[len(g) for g in groups]; rng=np.random.default_rng(seed); c=0
    for _ in range(n_perm):
        pp=rng.permutation(allv); idx=0; gs=[]
        for s in sizes: gs.append(pp[idx:idx+s]); idx+=s
        if max(_jt_stat(gs,o) for o in orders)>=obs: c+=1
    return (c+1)/(n_perm+1)
def directional_jt_general(per_fold_ordered,n_perm=10000,seed=RANDOM_STATE):
    groups=[np.asarray(g) for g in per_fold_ordered]
    if any(len(g)==0 for g in groups): return np.nan
    k=len(groups); obs=_jt_stat(groups,tuple(range(k)))
    allv=np.concatenate(groups); sizes=[len(g) for g in groups]; rng=np.random.default_rng(seed); c=0
    for _ in range(n_perm):
        pp=rng.permutation(allv); idx=0; gs=[]
        for s in sizes: gs.append(pp[idx:idx+s]); idx+=s
        if _jt_stat(gs,tuple(range(k)))>=obs: c+=1
    return (c+1)/(n_perm+1)
print("stats ready")

## Config engines + stat block + prep_window

In [ ]:
# Within-stratum quartile assignment
def prep_window(df, lo, hi, sev_col='apache_iva', band_width=10):
    d=df.dropna(subset=[sev_col,FREQ_COL]).copy()
    d[sev_col]=pd.to_numeric(d[sev_col],errors='coerce')
    d=d[(d[sev_col]>=lo)&(d[sev_col]<hi)].copy()
    bands=np.arange(lo, hi+band_width, band_width)
    d['_band']=pd.cut(d[sev_col], bins=bands, include_lowest=True)
    def _rank_quartile(x):                      # rank(method='first') -> equal sizes despite ties
        if len(x) < 8:
            return pd.Series([np.nan]*len(x), index=x.index)
        return pd.qcut(x.rank(method='first'), q=4, labels=QUARTILES)
    d['quartile']=d.groupby('_band', group_keys=False, observed=True)[FREQ_COL].transform(_rank_quartile)
    d=d.dropna(subset=['quartile']).copy()
    d['quartile']=d['quartile'].astype(str)     x
    return d.drop(columns='_band').reset_index(drop=True)

def run_quartile_config(d, model_kind, feature_cols, n_boot=2000):
    avail=[c for c in feature_cols if c in d.columns]
    strat=d['quartile'].astype(str)+'_'+d[MORTALITY].astype(str)
    skf=StratifiedKFold(N_FOLDS,shuffle=True,random_state=RANDOM_STATE)
    pool={q:([],[]) for q in QUARTILES}; perfold={q:[] for q in QUARTILES}; base_y,base_p=[],[]
    for tr,te in skf.split(d,strat):
        trf,tef=d.iloc[tr],d.iloc[te]
        train_df=trf if model_kind=='A' else trf[trf.quartile=='Q1']
        if train_df[MORTALITY].nunique()<2 or len(train_df)<20: continue
        med=train_df[avail].median()
        m=train_xgb(train_df[avail].fillna(med),train_df[MORTALITY])
        pl=fit_platt_holdout(m,train_df[avail].fillna(med),train_df[MORTALITY])
        for q in QUARTILES:
            tq=tef[tef.quartile==q]
            if len(tq)<5: continue
            pr=apply_platt(pl,m.predict_proba(tq[avail].fillna(med))[:,1])
            pool[q][0].extend(tq[MORTALITY].values); pool[q][1].extend(pr)
            if tq[MORTALITY].nunique()>=2: perfold[q].append(roc_auc_score(tq[MORTALITY].values,pr))
        prb=apply_platt(pl,m.predict_proba(tef[avail].fillna(med))[:,1])
        base_y.extend(tef[MORTALITY].values); base_p.extend(prb)
    aucs={q:boot_ci(*pool[q],n_boot=n_boot) for q in QUARTILES}
    base=roc_auc_score(base_y,base_p) if len(np.unique(base_y))>=2 else np.nan
    return {'aucs':aucs,'perfold':perfold,'pool':pool,'base_auc':base}

def run_race_config(d, model_kind, feature_cols, race_col='race', n_boot=1000):
    avail=[c for c in feature_cols if c in d.columns]
    groups=[g for g in ['White','Black','Hispanic/Latino','Asian'] if (d[race_col]==g).any()]
    strat=d['quartile'].astype(str)+'_'+d[MORTALITY].astype(str)
    skf=StratifiedKFold(N_FOLDS,shuffle=True,random_state=RANDOM_STATE)
    pool={g:([],[]) for g in groups}; perfold={g:[] for g in groups}
    for tr,te in skf.split(d,strat):
        trf,tef=d.iloc[tr],d.iloc[te]
        train_df=trf if model_kind=='A' else trf[trf.quartile=='Q1']
        if train_df[MORTALITY].nunique()<2 or len(train_df)<20: continue
        med=train_df[avail].median()
        m=train_xgb(train_df[avail].fillna(med),train_df[MORTALITY])
        pl=fit_platt_holdout(m,train_df[avail].fillna(med),train_df[MORTALITY])
        for g in groups:
            tg=tef[tef[race_col]==g]
            if len(tg)<5 or tg[MORTALITY].nunique()<2: continue
            pr=apply_platt(pl,m.predict_proba(tg[avail].fillna(med))[:,1])
            pool[g][0].extend(tg[MORTALITY].values); pool[g][1].extend(pr)
            perfold[g].append(roc_auc_score(tg[MORTALITY].values,pr))
    groups=[g for g in groups if len(pool[g][0])>=MIN_RACE_N and int(np.sum(pool[g][0]))>=MIN_RACE_DEATHS and len(np.unique(pool[g][0]))>=2]
    aucs={g:boot_ci(*pool[g],n_boot=n_boot) for g in groups}
    return {'aucs':aucs,'perfold':perfold,'pool':pool,'groups':groups}

def full_stat_block(cfg, keys, ordered):
    def fmt(ci): return f"{ci[0]:.3f} ({ci[1]:.3f}-{ci[2]:.3f})" if not np.isnan(ci[0]) else "NA"
    est=[k for k in keys if k in cfg['aucs']]
    for k in est: print(f"     {k:16s} {fmt(cfg['aucs'][k])}")
    pairs=list(combinations(est,2)); raw={}
    for a,b in pairs:
        ya,pa=np.array(cfg['pool'][a][0]),np.array(cfg['pool'][a][1])
        yb,pb=np.array(cfg['pool'][b][0]),np.array(cfg['pool'][b][1])
        if len(ya)>=5 and len(yb)>=5 and len(np.unique(ya))>=2 and len(np.unique(yb))>=2:
            raw[(a,b)]=delong_p(ya,pa,yb,pb)
    nsig=sum(1 for v in raw.values() if not np.isnan(v) and v*max(len(pairs),1)<0.05)
    pmax=perm_max_jt_general([cfg['perfold'][k] for k in est])
    line=f"     DeLong sig {nsig}/{len(pairs)} | perm-max JT p={pmax:.4f}"
    if ordered and len(est)>=3:
        m=[cfg['aucs'][k][0] for k in est]; rho,_=stats.spearmanr(range(len(m)),m)
        djt=directional_jt_general([cfg['perfold'][k] for k in est])
        line+=f" | directional JT p={djt:.4f} | Spearman rho={rho:+.2f} | Q1-Q4 d={m[0]-m[-1]:+.3f}"
    print(line)
print("engine ready")

## Four-configuration

In [ ]:
def four_config(care, df, feats_used, lo, hi):
    d_win=prep_window(df,lo,hi)
    d_full=df.dropna(subset=['apache_iva',FREQ_COL]).copy()
    d_full['apache_iva']=pd.to_numeric(d_full['apache_iva'],errors='coerce')
    d_full['quartile']=pd.qcut(d_full[FREQ_COL].rank(method='first'),4,labels=QUARTILES)
    d_full=d_full.dropna(subset=['quartile']).reset_index(drop=True)
    drace=d_full[d_full['race']!='Other'].copy()
    print(f"\n{'#'*72}\n# {care} — quartile APACHE [{lo},{hi}) n={len(d_win)} | race FULL n={len(drace)}\n{'#'*72}")
    comp=drace.groupby('race')[MORTALITY].agg(['size','sum'])
    print("  race (n / deaths):")
    for g,r in comp.iterrows(): print(f"     {g:16s} n={int(r['size']):4d} deaths={int(r['sum']):3d}")
    c2=run_quartile_config(d_win,'A',feats_used); c3=run_quartile_config(d_win,'B',feats_used)
    c1=run_race_config(drace,'A',feats_used);     c4=run_race_config(drace,'B',feats_used)
    print("\n  -- Config 1 — Conventional Model | RACE (expect uniform) --");      full_stat_block(c1,['White','Black','Hispanic/Latino','Asian'],False)
    print("\n  -- Config 2 — Conventional Model | QUARTILE (correlational signal) --"); full_stat_block(c2,QUARTILES,True)
    print("\n  -- Config 3 — TRM | QUARTILE *** degradation *** --"); full_stat_block(c3,QUARTILES,True)
    print("\n  -- Config 4 — TRM | RACE (expect uniform) --");      full_stat_block(c4,['White','Black','Hispanic/Latino','Asian'],False)
    return {'c1':c1,'c2':c2,'c3':c3,'c4':c4}
print("runner ready")

## Load CSVs (glucose, pain)

In [ ]:
def map_race(e):
    e=str(e).lower()
    if 'caucasian' in e or 'white' in e:  return 'White'
    if 'african' in e or 'black' in e:    return 'Black'
    if 'hispanic' in e or 'latino' in e:  return 'Hispanic/Latino'
    if 'asian' in e:                      return 'Asian'
    return 'Other'

def load_combined(fname):
    df=pd.read_csv(os.path.join(MIT,fname))
    df[MORTALITY]=df[MORTALITY].astype(int)
    df['race']=df['ethnicity'].map(map_race)
    for c in APS_PRIMARY:
        if c in df.columns:
            df[c]=pd.to_numeric(df[c],errors='coerce'); df.loc[df[c]<0,c]=np.nan
    return df

DATA={name:load_combined(f) for name,f in PHENOTYPES.items()}
for name,df in DATA.items():
    print(f"{name}: {len(df)} stays | interval {df[FREQ_COL].notna().sum()} | race {df['race'].value_counts().to_dict()}")

## Clinical-Regime Decomposition (locate window from data, before fixing)

In [ ]:
def cd_sweep(df, care, feature_cols, width=25, step=10, lo0=40, hi0=160, min_n=150, min_events=20):
    avail=[c for c in feature_cols if c in df.columns]
    d=df.dropna(subset=['apache_iva',FREQ_COL]).copy()
    d['apache_iva']=pd.to_numeric(d['apache_iva'],errors='coerce')
    rows=[]
    print(f"\n{'='*64}\n{care} — CD sweep (TRM Q1 vs Q4 across APACHE)\n{'='*64}")
    for lo in range(lo0,hi0-width+1,step):
        hi=lo+width
        b=d[(d['apache_iva']>=lo)&(d['apache_iva']<hi)].copy()
        if len(b)<min_n or b[MORTALITY].sum()<min_events: continue
        b['quartile']=pd.qcut(b[FREQ_COL].rank(method='first'),4,labels=QUARTILES)
        b=b.dropna(subset=['quartile']).reset_index(drop=True)
        strat=b['quartile'].astype(str)+'_'+b[MORTALITY].astype(str)
        if strat.value_counts().min()<N_FOLDS: continue
        skf=StratifiedKFold(N_FOLDS,shuffle=True,random_state=RANDOM_STATE)
        pool={q:([],[]) for q in QUARTILES}; perfold={q:[] for q in QUARTILES}
        for tr,te in skf.split(b,strat):
            trf,tef=b.iloc[tr],b.iloc[te]; q1=trf[trf.quartile=='Q1']
            if len(q1)<15 or q1[MORTALITY].nunique()<2: continue
            med=q1[avail].median()
            m=train_xgb(q1[avail].fillna(med),q1[MORTALITY])
            pl=fit_platt_holdout(m,q1[avail].fillna(med),q1[MORTALITY])
            for q in QUARTILES:
                tq=tef[tef.quartile==q]
                if len(tq)<5 or tq[MORTALITY].nunique()<2: continue
                pr=apply_platt(pl,m.predict_proba(tq[avail].fillna(med))[:,1])
                pool[q][0].extend(tq[MORTALITY].values); pool[q][1].extend(pr)
                perfold[q].append(roc_auc_score(tq[MORTALITY].values,pr))
        a={q:(roc_auc_score(pool[q][0],pool[q][1]) if len(np.unique(pool[q][0]))>1 else np.nan) for q in QUARTILES}
        gap=a['Q1']-a['Q4']; rho,_=stats.spearmanr([1,2,3,4],[a[q] for q in QUARTILES])
        djt=directional_jt_general([perfold[q] for q in QUARTILES],n_perm=2000)
        rows.append({'lo':lo,'hi':hi,'mid':lo+width/2,'n':len(b),'Q1':a['Q1'],'Q4':a['Q4'],'gap':gap,'rho':rho,'dirJT':djt})
        print(f"  APACHE {lo:3d}-{hi:3d} n={len(b):4d} | Q1={a['Q1']:.3f} Q4={a['Q4']:.3f} gap={gap:+.3f} | rho={rho:+.2f} dirJT={djt:.3f}")
    return pd.DataFrame(rows)

# uses the eICU-native feature set
cd_results={name:cd_sweep(df,name,APS_PRIMARY) for name,df in DATA.items()}

In [ ]:
import matplotlib.pyplot as plt
def plot_cd(res, care):
    if res.empty: print(f"{care}: no qualifying windows"); return
    fig,ax=plt.subplots(figsize=(8,5)); fig.patch.set_facecolor('white')
    ax.plot(res['mid'],res['Q1'],'-o',color='#1a6e9e',lw=2.3,ms=6,label='Q1 (best care)')
    ax.plot(res['mid'],res['Q4'],'--s',color='#c0392b',lw=2.3,ms=6,label='Q4 (worst care)')
    ax.fill_between(res['mid'],res['Q1'],res['Q4'],where=(res['Q1']>=res['Q4']),alpha=0.15,color='#1a6e9e')
    lo,hi=WINDOWS[care]; ax.axvspan(lo,hi,alpha=0.07,color='green')
    ax.axhline(0.5,color='#aaa',ls=':',lw=1)
    ax.set_xlabel('APACHE IVa (window midpoint)'); ax.set_ylabel('TRM AUROC')
    ax.set_title(f'{care} — CD sweep (shaded = window {lo}-{hi})',fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.25,ls='--'); ax.set_facecolor('#F8F9FA')
    for s in('top','right'): ax.spines[s].set_visible(False)
    plt.tight_layout(); plt.show()
for name in DATA: plot_cd(cd_results[name], name)

## Figures

In [ ]:
results_primary={}
for name,df in DATA.items():
    lo,hi=WINDOWS[name]
    results_primary[name]=four_config(name, df, APS_PRIMARY, lo, hi)

In [ ]:
import matplotlib.pyplot as plt
A_COL,B_COL='#2874A6','#C0392B'
def _err(ax,xs,cfg,keys,col,mk,label):
    m=np.array([cfg['aucs'][k][0] if k in cfg['aucs'] else np.nan for k in keys])
    lo=np.array([cfg['aucs'][k][1] if k in cfg['aucs'] else np.nan for k in keys])
    hi=np.array([cfg['aucs'][k][2] if k in cfg['aucs'] else np.nan for k in keys])
    v=~np.isnan(m)
    ax.errorbar(np.array(xs)[v],m[v],yerr=[(m-lo)[v],(hi-m)[v]],color=col,lw=2.3,marker=mk,ms=7,capsize=4,label=label)
def fig_four(res, care, arm):
    c1,c2,c3,c4=res['c1'],res['c2'],res['c3'],res['c4']
    fig,(axL,axR)=plt.subplots(1,2,figsize=(14,5.4)); fig.patch.set_facecolor('white')
    x=np.arange(4)
    _err(axL,x,c2,QUARTILES,A_COL,'o','Config 2 — Conventional Model'); _err(axL,x,c3,QUARTILES,B_COL,'s','Config 3 — TRM')
    axL.axhline(0.5,color='#aaa',ls=':',lw=1); axL.set_xticks(x); axL.set_xticklabels(QUARTILES)
    axL.set_xlabel('Care quartile (Q1 best -> Q4 worst)'); axL.set_ylabel('AUROC'); axL.set_ylim(0.30,0.92)
    axL.legend(fontsize=8.5,loc='lower left'); axL.grid(alpha=0.25,ls='--'); axL.set_facecolor('#F8F9FA')
    for s in('top','right'): axL.spines[s].set_visible(False)
    axL.set_title('Care axis — degradation (C2 vs C3)',fontsize=10,fontweight='bold')
    rk=['White','Black','Hispanic/Latino','Asian']; present=[g for g in rk if (g in c1['aucs']) or (g in c4['aucs'])]
    xr=np.arange(len(present))
    _err(axR,xr,c1,present,A_COL,'o','Config 1 — Conventional Model'); _err(axR,xr,c4,present,B_COL,'s','Config 4 — TRM')
    axR.axhline(0.5,color='#aaa',ls=':',lw=1); axR.set_xticks(xr); axR.set_xticklabels([p.replace('/Latino','') for p in present])
    axR.set_xlabel('Race group'); axR.set_ylabel('AUROC'); axR.set_ylim(0.30,0.92)
    axR.legend(fontsize=8.5,loc='lower left'); axR.grid(alpha=0.25,ls='--'); axR.set_facecolor('#F8F9FA')
    for s in('top','right'): axR.spines[s].set_visible(False)
    axR.set_title('Race axis — uniformity (C1 vs C4)',fontsize=10,fontweight='bold')
    lo,hi=WINDOWS[care]; fig.suptitle(f'{care} — APACHE [{lo},{hi}) — {arm}',fontsize=12,fontweight='bold',y=1.02)
    plt.tight_layout(w_pad=2.5); plt.show()

print("=== PRIMARY (eICU-native APACHE features) ===")
for name in DATA: fig_four(results_primary[name], name, 'primary: eICU-native APACHE features')

## Repeated-CV race uniformity, perm-max JT stability check

In [ ]:
def run_race_config_repeated(d_full, model_kind, feature_cols,
                             n_repeats=10, seeds=None):
    if seeds is None:
        seeds=[42,123,7,99,314,271,1000,2024,55,888][:n_repeats]
    avail=[c for c in feature_cols if c in d_full.columns]
    drace=d_full[d_full['race']!='Other'].copy()
    groups=[g for g in ['White','Black','Hispanic/Latino','Asian'] if (drace['race']==g).any()]
    perfold={g:[] for g in groups}
    for seed in seeds:
        skf=StratifiedKFold(N_FOLDS,shuffle=True,random_state=seed)
        strat=drace['quartile'].astype(str)+'_'+drace[MORTALITY].astype(str)
        for tr,te in skf.split(drace,strat):
            trf,tef=drace.iloc[tr],drace.iloc[te]
            train_df=trf if model_kind=='A' else trf[trf.quartile=='Q1']
            if train_df[MORTALITY].nunique()<2 or len(train_df)<20: continue
            med=train_df[avail].median()
            m=train_xgb(train_df[avail].fillna(med),train_df[MORTALITY])
            pl=fit_platt_holdout(m,train_df[avail].fillna(med),train_df[MORTALITY])
            for g in groups:
                tg=tef[tef.race==g]
                if len(tg)<5 or tg[MORTALITY].nunique()<2: continue
                pr=apply_platt(pl,m.predict_proba(tg[avail].fillna(med))[:,1])
                perfold[g].append(roc_auc_score(tg[MORTALITY].values,pr))
    elig=[g for g in groups if len(perfold[g])>=4]
    pmax=perm_max_jt_general([np.array(perfold[g]) for g in elig]) if len(elig)>=2 else np.nan
    print(f"  repeated-CV ({len(seeds)} seeds) race perm-max JT p={pmax:.4f}")
    for g in groups:
        v=perfold[g]
        if v: print(f"    {g:16s} n_est={len(v):3d} mean={np.mean(v):.3f} sd={np.std(v):.3f} "
                     f"min={np.min(v):.3f} max={np.max(v):.3f}")
    return pmax, perfold

def _prep_race_frame(df):
    d=df.dropna(subset=['apache_iva',FREQ_COL]).copy()
    d['apache_iva']=pd.to_numeric(d['apache_iva'],errors='coerce')
    d['quartile']=pd.qcut(d[FREQ_COL].rank(method='first'),4,labels=QUARTILES)
    return d.dropna(subset=['quartile'])

REPEATED_RACE={}
for name,df in DATA.items():
    dr=_prep_race_frame(df)
    print(f"\n=== {name} Config 1 (Conventional Model) — repeated-CV race ===")
    p1,_=run_race_config_repeated(dr,'A',APS_PRIMARY)
    print(f"=== {name} Config 4 (TRM) — repeated-CV race ===")
    p4,_=run_race_config_repeated(dr,'B',APS_PRIMARY)
    REPEATED_RACE[name]={'C1':p1,'C4':p4}

In [ ]:
# Cohort characteristics, KW severity balance, interval monotonicity 
from scipy import stats

def cohort_table(care, df):
    lo,hi=WINDOWS[care]
    d=prep_window(df,lo,hi)
    age=pd.to_numeric(d['age_years'],errors='coerce')
    female=d['gender'].astype(str).str.lower().str.startswith('f').mean()*100
    print(f"\n{'='*60}\n{care} — APACHE [{lo},{hi})\n{'='*60}")
    print(f"  N={len(d)}  mortality={d[MORTALITY].mean()*100:.1f}%  "
          f"median age={age.median():.0f} (IQR {age.quantile(.25):.0f}-{age.quantile(.75):.0f})  female={female:.1f}%")
    # KW severity balance across quartiles (validity check: severity decoupled from care)
    kw=stats.kruskal(*[pd.to_numeric(d[d.quartile==q]['apache_iva'],errors='coerce').dropna() for q in QUARTILES])
    bal='balanced' if kw.pvalue>=0.05 else 'IMBALANCED'
    print(f"  APACHE balance across Q1-Q4: Kruskal-Wallis P={kw.pvalue:.3f} ({bal})")
    print("    mean APACHE: " + "  ".join(f"{q}={pd.to_numeric(d[d.quartile==q]['apache_iva'],errors='coerce').mean():.1f}" for q in QUARTILES))
    # mean care interval (hours) monotonic Q1->Q4
    iv=[d[d.quartile==q][FREQ_COL].mean()/60.0 for q in QUARTILES]
    mono='monotonic increasing' if all(iv[i]<iv[i+1] for i in range(3)) else 'non-monotonic'
    print(f"  mean care interval (h): " + "  ".join(f"{q}={v:.1f}" for q,v in zip(QUARTILES,iv)) + f"  ({mono})")
    # per-quartile mortality / deaths
    print("  per-quartile n / deaths / mortality:")
    for q in QUARTILES:
        dq=d[d.quartile==q]
        print(f"    {q}: n={len(dq):4d} deaths={int(dq[MORTALITY].sum()):3d} mort={dq[MORTALITY].mean()*100:4.1f}%")
    # race composition in-window
    print("  race (in-window):", d['race'].value_counts().to_dict())

for name,df in DATA.items():
    cohort_table(name, df)

In [ ]:
# Calibration
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression

def _reliability(y, p, n_bins=8):
    y,p=np.asarray(y,float),np.asarray(p,float)
    if len(y)<20 or len(np.unique(y))<2: return np.array([]),np.array([])
    fp,mp=calibration_curve(y,p,n_bins=n_bins,strategy='quantile')     
    return mp, fp     # x=mean predicted, y=observed

def _ece(y, p, n_bins=8):       # quantile-binned ECE to match the curve
    y,p=np.asarray(y,float),np.asarray(p,float)
    if len(y)<20 or len(np.unique(y))<2: return np.nan
    import pandas as pd
    edges=np.quantile(p,np.linspace(0,1,n_bins+1)); edges[0]-=1e-9
    e=0.0
    for i in range(n_bins):
        m=(p>edges[i])&(p<=edges[i+1])
        if m.sum()==0: continue
        e+=abs(p[m].mean()-y[m].mean())*m.sum()/len(y)
    return e

from sklearn.model_selection import train_test_split as _cal_tts

def calibration_TRM(care, df, feats_used, calib='platt'):
    lo,hi=WINDOWS[care]; d=prep_window(df,lo,hi)
    avail=[c for c in feats_used if c in d.columns]
    strat=d['quartile'].astype(str)+'_'+d[MORTALITY].astype(str)
    skf=StratifiedKFold(N_FOLDS,shuffle=True,random_state=RANDOM_STATE)
    pool={q:([],[]) for q in QUARTILES}
    for tr,te in skf.split(d,strat):
        trf,tef=d.iloc[tr],d.iloc[te]; q1=trf[trf.quartile=='Q1']
        if len(q1)<25 or q1[MORTALITY].nunique()<2: continue
        med=q1[avail].median()
        # split Q1 training fold: 85% to train model, 15% held-out to fit calibrator
        Xtr,Xcal,ytr,ycal=_cal_tts(q1[avail].fillna(med), q1[MORTALITY],
                                   test_size=0.15, stratify=q1[MORTALITY], random_state=RANDOM_STATE)
        m=train_xgb(Xtr,ytr)
        raw_cal=m.predict_proba(Xcal)[:,1]                
        if calib=='platt':
            cal=LogisticRegression().fit(raw_cal.reshape(-1,1),ycal); f=lambda r: cal.predict_proba(r.reshape(-1,1))[:,1]
        else:
            cal=IsotonicRegression(out_of_bounds='clip').fit(raw_cal,ycal.values); f=lambda r: cal.predict(r)
        for q in QUARTILES:
            tq=tef[tef.quartile==q]
            if len(tq)<5: continue
            pr=f(m.predict_proba(tq[avail].fillna(med))[:,1])
            pool[q][0].extend(tq[MORTALITY].values); pool[q][1].extend(pr)
    eces={q:_ece(pool[q][0],pool[q][1]) for q in QUARTILES if len(pool[q][0])>=20}
    gaps={q:(np.mean(pool[q][0])-np.mean(pool[q][1])) for q in QUARTILES if len(pool[q][0])>=20}
    return pool, eces, gaps

def plot_calibration(care, df, feats_used):
    pool,eces,gaps=calibration_TRM(care,df,feats_used,'platt')
    fig,ax=plt.subplots(figsize=(6.5,6)); fig.patch.set_facecolor('white')
    cols={'Q1':'#1a6e9e','Q2':'#5ba3c9','Q3':'#e08a7a','Q4':'#c0392b'}
    ax.plot([0,1],[0,1],'k:',lw=1,label='perfect')
    for q in QUARTILES:
        if len(pool[q][0])<10: continue
        xs,ys=_reliability(pool[q][0],pool[q][1])
        ax.plot(xs,ys,'-o',color=cols[q],lw=2,ms=5,label=f"{q} (ECE={eces.get(q,float('nan')):.3f})")
    ax.set_xlabel('Predicted mortality'); ax.set_ylabel('Observed mortality')
    ax.set_title(f'{care} — TRM calibration by quartile (Platt)',fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.25,ls='--'); ax.set_facecolor('#F8F9FA')
    for s in('top','right'): ax.spines[s].set_visible(False)
    plt.tight_layout(); plt.show()
    _,eces_iso,_=calibration_TRM(care,df,feats_used,'isotonic')
    print(f"  {care} ECE Q1->Q4 (Platt):    " + " ".join(f"{q}={eces.get(q,float('nan')):.3f}" for q in QUARTILES))
    print(f"  {care} ECE Q1->Q4 (Isotonic): " + " ".join(f"{q}={eces_iso.get(q,float('nan')):.3f}" for q in QUARTILES))
    print(f"  {care} gap (actual-pred) Q1->Q4: " + " ".join(f"{q}={gaps.get(q,float('nan')):+.3f}" for q in QUARTILES))

for name,df in DATA.items():
    plot_calibration(name, df, APS_PRIMARY) 

In [ ]:
# Results
rows=[]
def grab(care, res, arm):
    for cfg,axis,keys,ordered in [('C1','race',['White','Black','Hispanic/Latino','Asian'],False),
                                   ('C2','care',QUARTILES,True),
                                   ('C3','care',QUARTILES,True),
                                   ('C4','race',['White','Black','Hispanic/Latino','Asian'],False)]:
        c=res[{'C1':'c1','C2':'c2','C3':'c3','C4':'c4'}[cfg]]
        est=[k for k in keys if k in c['aucs']]
        aucs="; ".join(f"{k.split('/')[0]} {c['aucs'][k][0]:.3f} ({c['aucs'][k][1]:.3f}-{c['aucs'][k][2]:.3f})" for k in est)
        from itertools import combinations
        pairs=list(combinations(est,2)); nsig=0
        for a,b in pairs:
            ya,pa=np.array(c['pool'][a][0]),np.array(c['pool'][a][1]); yb,pb=np.array(c['pool'][b][0]),np.array(c['pool'][b][1])
            if len(ya)>=5 and len(yb)>=5 and len(np.unique(ya))>=2 and len(np.unique(yb))>=2:
                p=delong_p(ya,pa,yb,pb)
                if not np.isnan(p) and p*max(len(pairs),1)<0.05: nsig+=1
        pmax=perm_max_jt_general([c['perfold'][k] for k in est])
        row={'phenotype':care,'arm':arm,'config':cfg,'axis':axis,'aucs':aucs,
             'DeLong_sig':f"{nsig}/{len(pairs)}",'permJT':round(pmax,4)}
        if ordered and len(est)>=3:
            m=[c['aucs'][k][0] for k in est]; rho,_=stats.spearmanr(range(len(m)),m)
            row['dirJT']=round(directional_jt_general([c['perfold'][k] for k in est]),4)
            row['rho']=round(rho,2); row['Q1_Q4']=round(m[0]-m[-1],3)
        rows.append(row)

for name in DATA:
    grab(name, results_primary[name], 'primary')

summary=pd.DataFrame(rows)
summary.to_csv('eicu_results_summary.csv', index=False)
print(summary.to_string(index=False))
print("\nsaved -> eicu_results_summary.csv")